In [1]:
import os, re, random
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report

In [2]:
if os.path.exists('spam.csv'):
    df = pd.read_csv('spam.csv', encoding='latin-1').iloc[:, :2]
    df.columns = ['labels', 'message']
else:
    random.seed(1)
    spam = ["Free entry to win a cash prize call now", "Congratulations you won a free gift claim now",
            "URGENT your account is suspended click link to verify", "Win free tickets text WIN to 80082",
            "Cheap loans approved instantly apply now", "You have been selected for a free holiday",
            "Claim your free reward click here now", "Limited offer buy now get 50% discount free shipping",
            "Lottery winner! send bank details to claim prize", "Get paid fast work from home earn cash now"]
    ham = ["Are we meeting for lunch tomorrow", "Please find the project report attached",
           "Can you call me when you reach home", "Meeting rescheduled to 3 pm in the conference room",
           "Thanks for the notes, I will review them tonight", "Happy birthday! see you at the party",
           "Did you finish the assignment for the lab", "Let's discuss the schedule for next week",
           "I will be late, start the class without me", "Dinner at my place on Saturday?"]
    extra_s = ["now", "today", "!!!", "limited time", "click here"]
    extra_h = ["thanks", "regards", "see you", "tomorrow", "ok"]
    rows = [("spam", s + " " + random.choice(extra_s)) for s in spam for _ in range(6)] + \
           [("ham",  h + " " + random.choice(extra_h)) for h in ham  for _ in range(6)]
    df = pd.DataFrame(rows, columns=['labels', 'message'])

In [3]:
print("Dataset shape:", df.shape)
print(df['labels'].value_counts(), "\n")

Dataset shape: (5572, 2)
labels
ham     4825
spam     747
Name: count, dtype: int64 



In [4]:
# Pre-processing: lowercase, strip punctuation; stop-words removed by the vectorizer
clean = lambda t: re.sub(r'[^a-z0-9\s]', ' ', t.lower())
df['clean'] = df['message'].apply(clean)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean'], df['labels'], test_size=0.25, stratify=df['labels'], random_state=42)

In [6]:
vec = TfidfVectorizer(stop_words='english')
Xtr, Xte = vec.fit_transform(X_train), vec.transform(X_test)

In [7]:
# Tune K by trial
for k in [1, 3, 5, 7]:
    acc = accuracy_score(y_test, KNeighborsClassifier(n_neighbors=k, metric='euclidean').fit(Xtr, y_train).predict(Xte))
    print(f"K={k}: accuracy={acc:.3f}")

knn = KNeighborsClassifier(n_neighbors=3, metric='euclidean').fit(Xtr, y_train)
pred = knn.predict(Xte)

K=1: accuracy=0.949
K=3: accuracy=0.924
K=5: accuracy=0.910
K=7: accuracy=0.903


In [8]:
tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=['ham', 'spam']).ravel()
print("\nConfusion matrix [ham, spam]:\n", confusion_matrix(y_test, pred, labels=['ham', 'spam']))
print(f"TP={tp} TN={tn} FP={fp} FN={fn}")
print(f"Accuracy    : {(tp+tn)/(tp+tn+fp+fn):.3f}")
print(f"Precision   : {tp/(tp+fp):.3f}")
print(f"Sensitivity : {tp/(tp+fn):.3f}")
print(f"Specificity : {tn/(tn+fp):.3f}")
print("\n", classification_report(y_test, pred))

for m in ["Congratulations! You won a free prize, click now", "Please send me the lab report tomorrow"]:
    print(f"'{m}' ->", knn.predict(vec.transform([clean(m)]))[0])


Confusion matrix [ham, spam]:
 [[1206    0]
 [ 106   81]]
TP=81 TN=1206 FP=0 FN=106
Accuracy    : 0.924
Precision   : 1.000
Sensitivity : 0.433
Specificity : 1.000

               precision    recall  f1-score   support

         ham       0.92      1.00      0.96      1206
        spam       1.00      0.43      0.60       187

    accuracy                           0.92      1393
   macro avg       0.96      0.72      0.78      1393
weighted avg       0.93      0.92      0.91      1393

'Congratulations! You won a free prize, click now' -> ham
'Please send me the lab report tomorrow' -> ham
